In [80]:
import pandas as pd

link_stream = pd.read_csv('data/link_stream.txt', delimiter=' ', names=['source', 'target', 'timestamp'])

In [81]:
import pandas as pd


nodes = pd.concat([link_stream['source'], link_stream['target']]).unique()

node2id = {node: idx for idx, node in enumerate(nodes)}

link_stream['source'] = link_stream['source'].map(node2id)
link_stream['target'] = link_stream['target'].map(node2id)
link_stream['idx'] = range(len(link_stream))

In [82]:
link_stream.to_csv('data/link_stream.csv', index=False)

In [ ]:
import random

class AliasSampler:
    def __init__(self, probs):
        """
        probs: dict(node -> probability)
        """
        self.nodes = list(probs.keys())
        p = list(probs.values())
        n = len(p)

        # 归一化（防止浮点误差）
        total = sum(p)
        p = [x / total for x in p]

        self.prob = [0] * n
        self.alias = [0] * n

        small = []
        large = []

        scaled_p = [x * n for x in p]

        for i, sp in enumerate(scaled_p):
            if sp < 1:
                small.append(i)
            else:
                large.append(i)

        while small and large:
            s = small.pop()
            l = large.pop()

            self.prob[s] = scaled_p[s]
            self.alias[s] = l

            scaled_p[l] = scaled_p[l] - (1 - scaled_p[s])
            if scaled_p[l] < 1:
                small.append(l)
            else:
                large.append(l)

        for i in large:
            self.prob[i] = 1
        for i in small:
            self.prob[i] = 1

    def sample(self):
        """返回一个节点"""
        i = random.randint(0, len(self.nodes) - 1)
        if random.random() < self.prob[i]:
            return self.nodes[i]
        else:
            return self.nodes[self.alias[i]]
        
"""
probs = {v: deg / (2 * m) for v, deg in degree_dict.items()}

sampler = AliasSampler(probs)

# 采样 10 次：
samples = [sampler.sample() for _ in range(10)]
print(samples)
"""

'\nprobs = {v: deg / (2 * m) for v, deg in degree_dict.items()}\n\nsampler = AliasSampler(probs)\n\n# 采样 10 次：\nsamples = [sampler.sample() for _ in range(10)]\nprint(samples)\n'

In [42]:
import torch

class StateManager: 
    def __init__(self, num_nodes, num_communities, device='cpu'):
        self.num_nodes = num_nodes
        self.num_communities = num_communities
        self.device = device
        # Static parameters
        self.global_degree = torch.zeros(num_nodes, device=device, dtype=torch.float32)
        # Initialize dynamic states
        self.node_lifespans = torch.zeros(num_nodes, device=device, dtype=torch.float32)
        self.total_duration = 1.0
        self.m = 0.0  # total number of links
        self.T_context = None
        self.T_buffer = torch.zeros((num_nodes, num_communities), device=device, dtype=torch.float32)
        self.last_seen = torch.full((num_nodes, num_communities), -1.0, device=device)
        self.last_prob = torch.ones((num_nodes, num_communities), device=device, dtype=torch.float32) / num_communities


    def pre_scan_data(self, link_stream):
        self.m = float(len(link_stream))

        all_nodes = pd.concat([link_stream['source'], link_stream['target']], ignore_index=True)
        all_nodes_t = torch.tensor(all_nodes.values, dtype=torch.long, device=self.device)
        self.global_degree = torch.bincount(all_nodes_t, minlength=self.num_nodes).float()

        t_max = link_stream['timestamp'].max()
        t_min = link_stream['timestamp'].min()
        total_duration = t_max - t_min

        print(f"T_duration = {total_duration}")
        self.total_duration = float(total_duration) if total_duration > 0 else 1.0
        print(f"Total links: {int(self.m)}")

        # compute node lifespans
        sources = link_stream[['source', 'timestamp']].rename(columns={'source': 'node'})
        targets = link_stream[['target', 'timestamp']].rename(columns={'target': 'node'})
        all_events = pd.concat([sources, targets], ignore_index=True)
        
        node_stats = all_events.groupby('node')['timestamp'].agg(['min', 'max'])
        
        epsilon = 1.0 
        lifespans = (node_stats['max'] - node_stats['min']).clip(lower=epsilon)
        
        # 填充 Tensor
        active_ids = torch.tensor(node_stats.index.values, device=self.device, dtype=torch.long)
        active_vals = torch.tensor(lifespans.values, device=self.device, dtype=torch.float32)
        
        self.node_lifespans.fill_(epsilon)
        self.node_lifespans[active_ids] = active_vals
        print(f"Max Lifespan: {self.node_lifespans.max().item()}")


    def init_context(self):
        """
        [Step 1] 在 pre_scan 后，训练开始前调用。
        使用生命周期初始化 T_Context。
        """
        # 初始化假设：节点把它的生命周期平均分配给所有社区
        # Shape: [N] -> [N, 1] -> [N, K]
        avg_duration = self.node_lifespans.unsqueeze(1) / self.num_communities
        self.T_context = avg_duration.repeat(1, self.num_communities)
        
        self.T_buffer.zero_()
        self.last_seen.fill_(-1.0)
        self.last_prob.fill_(1.0 / self.num_communities)
    
    def on_epoch_start(self):
        """每个 Epoch 开始时调用：清空累加器，重置时间"""
        self.T_buffer.zero_()
        self.last_seen.fill_(-1.0)
        # 注意：Last_Prob 不重置，继承上一轮学到的先验分布

    def get_context_for_loss(self, node_ids):
        """
        [Read] 在计算 Loss 时调用
        返回: 节点的历史总时长 (Detach, 无梯度)
        """
        return self.T_context[node_ids]

    def update_step(self, node_ids, new_probs, current_time):
        """
        [Write] 在 Optimizer.step() 之后调用
        逻辑：Sample-and-Hold 差分累积
        """
        node_ids = node_ids.to(self.device)
        new_probs = new_probs.detach().to(self.device)
        current_time = current_time.to(self.device)
        
        # 1. 获取上次状态
        prev_times = self.last_seen[node_ids]
        prev_probs = self.last_prob[node_ids]
        
        # 2. 识别非首次出现的节点
        # 只有之前出现过 (prev_time != -1)，才有"空窗期"需要填补
        mask = (prev_times != -1)
        
        if mask.any():
            valid_nodes = node_ids[mask]
            
            # 计算时间差 delta_t
            delta_t = current_time[mask] - prev_times[mask]
            # 确保时间非负 (防止数据乱序)
            delta_t = delta_t.clamp(min=0)
            
            # 计算时长增量: 旧分布 * 时间差
            # [M, 1] * [M, K]
            duration_inc = delta_t.unsqueeze(1) * prev_probs[mask]
            
            # 累加到 Buffer
            # 使用 index_add_ 处理 batch 内可能的重复节点
            self.T_buffer.index_add_(0, valid_nodes, duration_inc)

        # 3. 无论是否首次出现，都更新状态为当前值
        self.last_seen[node_ids] = current_time
        self.last_prob[node_ids] = new_probs

    def on_epoch_end(self):
        """
        Epoch 结束时调用：交换 Buffer
        """
        # 将本轮统计好的 T_Buffer 晋升为下一轮的 T_Context
        # 加极小值防止 sqrt(0) 梯度问题
        self.T_context = self.T_buffer.clone() + 1e-6
        print(f"Epoch 状态刷新。最大单节点累积时长: {self.T_context.max().item():.2f}")

In [ ]:
import torch

class GlobalDegreeSampler:
    def __init__(self, global_degrees):
        """
        初始化采样器
        global_degrees: Tensor [N], 也就是 state_mgr.global_degree
        """
        self.num_nodes = len(global_degrees)
        self.degrees = global_degrees.float()
        self.weights = self.degrees

    def sample(self, num_samples):
        """
        采样 num_samples 个负样本节点 ID
        返回: Tensor [num_samples]
        """
        # torch.multinomial 实现了加权无放回/有放回采样
        # replacement=True: 允许重复采样（在大图中通常没问题且更快）
        neg_samples = torch.multinomial(self.weights, num_samples, replacement=True)
        return neg_samples

In [43]:
node_set = set(pd.concat([link_stream['source'], link_stream['target']], ignore_index=True))
num_nodes = len(node_set)
print(f'num_nodes = {num_nodes} ')
num_comms = 5
state_mgr = StateManager(num_nodes, num_comms, device='cpu')
state_mgr.pre_scan_data(link_stream)
state_mgr.init_context()

num_nodes = 986 
T_duration = 69459254
Total links: 332334
Max Lifespan: 69440728.0


In [46]:
print(state_mgr.T_context)
print(state_mgr.T_context.shape)

tensor([[1.3850e+07, 1.3850e+07, 1.3850e+07, 1.3850e+07, 1.3850e+07],
        [1.3888e+07, 1.3888e+07, 1.3888e+07, 1.3888e+07, 1.3888e+07],
        [1.3850e+07, 1.3850e+07, 1.3850e+07, 1.3850e+07, 1.3850e+07],
        ...,
        [1.1963e+05, 1.1963e+05, 1.1963e+05, 1.1963e+05, 1.1963e+05],
        [2.0000e-01, 2.0000e-01, 2.0000e-01, 2.0000e-01, 2.0000e-01],
        [2.0000e-01, 2.0000e-01, 2.0000e-01, 2.0000e-01, 2.0000e-01]])
torch.Size([986, 5])


In [ ]:
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader

class LinkStreamDataset(Dataset):
    def __init__(self, df):
        """
        df: 包含 u, v, t 的 DataFrame。
        """
        # 1. 存数据 (Numpy 格式，极快)
        self.src = df['source'].values.astype(np.int64)
        self.dst = df['target'].values.astype(np.int64)
        self.times = df['timestamp' ].values.astype(np.float32)
        self.edge_idxs = df.index.values.astype(np.int64)

    def __len__(self):
        return len(self.src)

    def __getitem__(self, idx):
        return self.src[idx], self.dst[idx], self.times[idx], self.edge_idxs[idx]

def tgn_collate_fn(batch):
    """
    自定义整理函数：把 [(u1,v1,t1,i1), (u2,v2,t2,i2)] 变成 4 个 Tensor
    """
    # zip(*batch) 会把 list of tuples 转置为 tuple of lists
    src, dst, t, edge_idx = zip(*batch)
    
    return (
        torch.tensor(src, dtype=torch.long),
        torch.tensor(dst, dtype=torch.long),
        torch.tensor(t, dtype=torch.float32),
        torch.tensor(edge_idx, dtype=torch.long)
    )

dataset = LinkStreamDataset(link_stream)
dataloader = DataLoader(dataset, batch_size=500, shuffle=False, collate_fn=tgn_collate_fn)

In [ ]:
import math
import logging
import time
import sys
import argparse
import torch
import numpy as np
import pickle
from pathlib import Path

from tgn.evaluation.evaluation import eval_edge_prediction
from tgn.model.tgn import TGN
from tgn.utils.utils import EarlyStopMonitor, RandEdgeSampler, get_neighbor_finder
from tgn.utils.my_data import get_data, compute_time_statistics

NUM_EPOCHS = 3
BATCH_SIZE = 100
NUM_NEIGHBORS = 10
NUM_NEG = 1
NUM_HEADS = 2
DROP_OUT = 0.1
NUM_LAYER = 2
LEARNING_RATE = 0.0001
NODE_DIM = 16
TIME_DIM = 16
USE_MEMORY = True
MESSAGE_DIM = 100
MEMORY_DIM = 172
device = 'mps'


prefix = 'max_lmod'
data='link_stream'

Path("./saved_models/").mkdir(parents=True, exist_ok=True)
Path("./saved_checkpoints/").mkdir(parents=True, exist_ok=True)
MODEL_SAVE_PATH = f'./saved_models/{prefix}-{data}.pth'
get_checkpoint_path = lambda \
    epoch: f'./saved_checkpoints/{prefix}-{data}-{epoch}.pth'


logging.basicConfig(level=logging.INFO)
logger = logging.getLogger()
logger.setLevel(logging.DEBUG)
Path("log/").mkdir(parents=True, exist_ok=True)
fh = logging.FileHandler('log/{}.log'.format(str(time.time())))
fh.setLevel(logging.DEBUG)
ch = logging.StreamHandler()
ch.setLevel(logging.WARN)
formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
fh.setFormatter(formatter)
ch.setFormatter(formatter)
logger.addHandler(fh)
logger.addHandler(ch)


In [114]:
from tgn.utils.my_data import get_data  
import importlib
import sys
importlib.reload(sys.modules['tgn.utils.my_data'])

data = 'link_stream'

node_feat, edge_feat, full_data = get_data(data)
max_idx = max(full_data.unique_nodes)

cannot find (./data/link_stream.npy), use zero-vector (dim=16)...
cannot find node feature: ./data/link_stream_node.npy), use zero vector(dim=16)...
The dataset has 332334 interactions, involving 986 different nodes


In [111]:
from tgn.utils.utils import EarlyStopMonitor, get_neighbor_finder, MLP

ngh_finder = get_neighbor_finder(full_data, uniform=True, max_node_idx=max_idx)

In [99]:
from tgn.utils.my_data import get_data, compute_time_statistics
import importlib
import sys
importlib.reload(sys.modules['tgn.utils.my_data'])

mean_time_shift, std_time_shift= \
  compute_time_statistics(
      full_data.sources, 
      full_data.destinations, 
      full_data.timestamps
  )


In [124]:
from tgn.model.my_tgn import TGN
from pathlib import Path


NUM_EPOCHS = 3
BATCH_SIZE = 100
NUM_NEIGHBORS = 10
NUM_NEG = 1
NUM_HEADS = 2
DROP_OUT = 0.1
NUM_LAYER = 2
LEARNING_RATE = 0.0001
NODE_DIM = 16
TIME_DIM = 16
USE_MEMORY = True
MESSAGE_DIM = 100
MEMORY_DIM = 172
device = 'mps'


aggregator = 'last'
memory_update_at_end = True
embedding_module = 'graph_attention'
message_function = 'mlp'

prefix = 'email'
n_runs = 2
for i in range(n_runs):
    results_path = "results/{}_community_{}.pkl".format(prefix, i) if i > 0 else "results/{}.pkl".format(prefix)
    Path("results/").mkdir(parents=True, exist_ok=True)

    tgn = TGN(
        neighbor_finder=ngh_finder,
        node_features=node_feat,
        edge_features=edge_feat,
        device=device,
        n_layers=NUM_LAYER,
        n_heads=NUM_HEADS,
        dropout=DROP_OUT,
        use_memory=USE_MEMORY,
        message_dimension=MESSAGE_DIM,
        memory_dimension=MEMORY_DIM,
        n_neighbors=NUM_NEIGHBORS,
        mean_time_shift=mean_time_shift,
        std_time_shift=std_time_shift,
        use_destination_embedding_in_message=True,
        use_source_embedding_in_message=True,

        memory_update_at_start=not memory_update_at_end,
        embedding_module_type=embedding_module,
        message_function=message_function,
        aggregator_type=aggregator, 

    ).to(device)

    optimizer = torch.optim.Adam(tgn.parameters(), lr=LEARNING_RATE)

In [123]:
import math 
import logging
import time


logging.basicConfig(level=logging.INFO)
logger = logging.getLogger()
logger.setLevel(logging.DEBUG)
Path("log/").mkdir(parents=True, exist_ok=True)
fh = logging.FileHandler('log/{}.log'.format(str(time.time())))
fh.setLevel(logging.DEBUG)
ch = logging.StreamHandler()
ch.setLevel(logging.WARN)
formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
fh.setFormatter(formatter)
ch.setFormatter(formatter)
logger.addHandler(fh)
logger.addHandler(ch)


Path("./saved_models/").mkdir(parents=True, exist_ok=True)
Path("./saved_checkpoints/").mkdir(parents=True, exist_ok=True)
MODEL_SAVE_PATH = f'./saved_models/{prefix}-{data}.pth'
get_checkpoint_path = lambda \
    epoch: f'./saved_checkpoints/{prefix}-{data}-{epoch}.pth'


num_instance = len(full_data.sources)
num_batches = math.ceil(len(full_data.sources) / BATCH_SIZE)
logger.debug(f'num_batches: {num_batches}')

DEBUG:root:num_batches: 3324


In [ ]:
for epoch in range(NUM_EPOCHS):
    state_mgr.on_epoch_start()
    if tgn.use_memory:
        tgn.memory.__init_memory__()

    tgn.set_neighbor_finder(ngh_finder)

    logger.info(f'Starting epoch {epoch}')
    for k in range(num_batches):
        


        total_loss = 0.0
        optimizer.zero_grad()
        

In [100]:
print(mean_time_shift, std_time_shift)

61462.57031931731 1539911.081142258
